# 프로토콜 v1.1 — E2c 채점: CruiseBench 직접 실행 결과와 PI-BDP의 짝지은 비교 (탐색 분석, CPU)

E2c 실행 노트북이 저장한 창별 예측과 PI-BDP의 고정 예측을 **같은 시험 창**에서 비교합니다.

- **주 채점 (v1.1 4.1–4.2절):** CruiseBench 순항 창, η = 5% 상한 레이블. PI-BDP는 사이클 t의 창에 사이클 t − 1 뒤의 예측 − 1을 씁니다. 각 엔진의 첫 사이클 창은 **모든 모델에서** 뺍니다. 모든 예측에 [0, 상한] 범위를 똑같이 적용합니다. 비교 모델은 시드 평균 예측을 씁니다.
- **검정 (6절):** PI-BDP V0 − 각 모델의 통합 RMSE 차이를 계층 붓스트랩(서브셋 → 엔진, 100,000회)으로. 95% 신뢰구간과 보정하지 않은 p값.
- **서술 (9절):** 신뢰구간이 비교 모델 쪽으로 0을 벗어나면 "덜 정확", 0을 포함하면 "비슷", PI-BDP 쪽으로 벗어나면 "더 정확".
- **보조 채점 (4.3절):** 상한 없는 사이클 단위. 비교 모델은 한 사이클 안의 창 예측을 평균. 상한 레이블로 학습한 모델에 불리하므로 기술 통계로만 냅니다.

**실행 전:** `RUN_COMMIT_URL`에 이 노트북을 커밋한 주소를 넣으세요. 런타임은 CPU면 됩니다.

In [ ]:
import os, sys, json, glob, time, hashlib, re
import numpy as np, pandas as pd
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/PI-BDP_NCMAPSS'
else:
    ROOT = os.environ.get('NCM_ROOT', './ncm')
IN3 = f'{ROOT}/phase3'; RUN = f'{ROOT}/addendum_v11/e2c_cruise'; OUT = f'{ROOT}/addendum_v11'
RUN_COMMIT_URL = ''
assert re.match(r'https://github\.com/.+/commit/[0-9a-f]{7,40}$', RUN_COMMIT_URL), '이 노트북을 먼저 커밋하고 주소를 넣으세요.'
SUBSETS = ['DS01-005', 'DS02-006', 'DS03-012', 'DS04', 'DS05', 'DS06', 'DS07', 'DS08a-009', 'DS08c-008']
CAPS = dict(zip(SUBSETS, [38, 34, 34, 43, 38, 41, 39, 34, 28]))
KW = ['subset', 'unit', 'cycle', 'win']
dec = json.load(open(f'{RUN}/seed_decision.json')); N_SEEDS = dec['n_seeds']
print('B단계 시드 수', N_SEEDS, '|', dec)

In [ ]:
# === 창별 예측 불러오기 ===
P = {}
for m in ['LSTM', 'GRU', 'TCN', 'TSMixer']:
    fs = [f'{RUN}/preds/{s}_{m}_s{sd}.csv.gz' for s in SUBSETS for sd in range(1, N_SEEDS + 1)]
    miss = [f for f in fs if not os.path.exists(f)]; assert not miss, f'{m}: 실행이 끝나지 않은 파일 {len(miss)}개'
    P[m] = pd.concat([pd.read_csv(f) for f in fs], ignore_index=True)
ref = P['TSMixer'][P['TSMixer'].seed == 1][KW + ['label_capped']].sort_values(KW).reset_index(drop=True)
# 레이블 확인: 원 코드의 상한 레이블 = min(EOL - cycle, cap)
V0 = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f'{IN3}/test_PI-BDP_V0_*.csv.gz'))], ignore_index=True)[['subset', 'unit', 'cycle', 'q500', 'true']]
eol = (V0.cycle + V0.true).groupby([V0.subset, V0.unit]).max().rename('eol').reset_index()
ref = ref.merge(eol, on=['subset', 'unit'], how='left')
assert ref.eol.notna().all(), '시험 엔진이 맞지 않음'
ref['true'] = ref.eol - ref.cycle; ref['cap'] = ref.subset.map(CAPS)
assert np.allclose(ref.label_capped, np.minimum(ref.true, ref.cap)), '상한 레이블이 EOL - cycle과 맞지 않음'
print('시험 창', len(ref))

In [ ]:
# === 같은 창에 모든 모델 놓기 (v1.1 4.2절) ===
first = ref.cycle == ref.groupby(['subset', 'unit']).cycle.transform('min')
B = ref[~first].copy(); n_first = int(first.sum())
prev = V0.assign(cycle=V0.cycle + 1)[['subset', 'unit', 'cycle', 'q500']]
B = B.merge(prev, on=['subset', 'unit', 'cycle'], how='left'); assert B.q500.notna().all(), 't-1 예측이 없는 창이 있음'
B['label'] = B.label_capped
PRED = {'PI-BDP V0': np.clip(B.q500.values - 1, 0, B.cap.values)}
for m, df in P.items():
    mean = df.groupby(KW, as_index=False).pred.mean()
    x = B[KW].merge(mean, on=KW, how='left'); assert x.pred.notna().all()
    PRED[m] = np.clip(x.pred.values, 0, B.cap.values)
print('채점 창', len(B), '| 뺀 첫 사이클 창', n_first)

In [ ]:
# === 지표 ===
def rmse(e): return float(np.sqrt(np.mean(np.square(e))))
rows = []
for m, p in PRED.items():
    e = p - B.label.values; st = pd.Series(e).groupby(B.subset.values).apply(lambda v: rmse(v))
    rows.append(dict(model=m, pooled_rmse=rmse(e), mean_subset_rmse=float(st.mean()), **{s: float(st[s]) for s in SUBSETS}))
for m, df in P.items():   # 시드별 (기술통계)
    r = []
    for sd, g in df.groupby('seed'):
        x = B[KW].merge(g[KW + ['pred']], on=KW, how='left')
        r.append(rmse(np.clip(x.pred.values, 0, B.cap.values) - B.label.values))
    rows.append(dict(model=f'{m} (per-seed mean)', pooled_rmse=float(np.mean(r)), seed_sd=float(np.std(r, ddof=1)) if len(r) > 1 else np.nan))
TAB = pd.DataFrame(rows); TAB.to_csv(f'{OUT}/e2c_scores.csv', index=False)
print(TAB.round(3).to_string(index=False))

In [ ]:
# === 계층 붓스트랩 (서브셋 → 엔진), PI-BDP V0 - 각 모델 ===
def hier_boot(ea, eb, subset, unit, n_boot=100000, seed=1):
    U = pd.DataFrame({'subset': subset, 'unit': unit, 'ea': ea ** 2, 'eb': eb ** 2}).groupby(['subset', 'unit']).agg(
        sa=('ea', 'sum'), sb=('eb', 'sum'), n=('ea', 'size')).reset_index()
    subs = sorted(U.subset.unique()); rng = np.random.default_rng(seed)
    per = [U[U.subset == s][['sa', 'sb', 'n']].values for s in subs]
    SA = np.zeros(n_boot); SB = np.zeros(n_boot); N = np.zeros(n_boot)
    pick = rng.integers(0, len(subs), (n_boot, len(subs)))
    for j in range(len(subs)):
        for si, arr in enumerate(per):
            r = np.flatnonzero(pick[:, j] == si)
            if not len(r): continue
            tot = arr[rng.integers(0, len(arr), (len(r), len(arr)))].sum(1)
            SA[r] += tot[:, 0]; SB[r] += tot[:, 1]; N[r] += tot[:, 2]
    d = np.sqrt(SA / N) - np.sqrt(SB / N)
    obs = float(np.sqrt(U.sa.sum() / U.n.sum()) - np.sqrt(U.sb.sum() / U.n.sum()))
    return dict(diff=obs, ci_lo=float(np.percentile(d, 2.5)), ci_hi=float(np.percentile(d, 97.5)),
                p_boot=float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))))
def wording(r):
    if r['ci_lo'] > 0: return 'PI-BDP less accurate'
    if r['ci_hi'] < 0: return 'PI-BDP more accurate (exploratory)'
    return 'comparable'
ea = PRED['PI-BDP V0'] - B.label.values; tests = []
for m in ['LSTM', 'GRU', 'TCN', 'TSMixer']:
    r = hier_boot(ea, PRED[m] - B.label.values, B.subset.values, B.unit.values)
    tests.append(dict(comparison=f'V0 - {m}', **r, wording=wording(r)))
TESTS = pd.DataFrame(tests); TESTS.to_csv(f'{OUT}/e2c_tests.csv', index=False)
print(TESTS.round(4).to_string(index=False))

In [ ]:
# === 보조: 상한 없는 사이클 단위 (v1.1 4.3절, 기술통계) ===
cyc = V0.rename(columns={'q500': 'PI-BDP V0'})[['subset', 'unit', 'cycle', 'true', 'PI-BDP V0']]
for m, df in P.items():
    cm = df.groupby(['subset', 'unit', 'cycle'], as_index=False).pred.mean().rename(columns={'pred': m})
    cyc = cyc.merge(cm, on=['subset', 'unit', 'cycle'], how='inner')
SEC = pd.DataFrame([dict(model=m, cycle_rmse_uncapped=rmse(cyc[m] - cyc.true), n_cycles=len(cyc)) for m in ['PI-BDP V0', 'LSTM', 'GRU', 'TCN', 'TSMixer']])
SEC.to_csv(f'{OUT}/e2c_cycle_level.csv', index=False); print(SEC.round(3).to_string(index=False))

In [ ]:
# === 요약 + 해시 ===
txt = '\n'.join([f"Addendum v1.1 E2c scoring (exploratory) {time.strftime('%Y-%m-%d %H:%M:%S')}  run={RUN_COMMIT_URL}",
    f"Seeds per model: {N_SEEDS}; windows scored {len(B)}; first-cycle windows excluded {n_first}",
    '', 'Main scoring (eta=5% caps, CruiseBench cruise windows, final-epoch models, seed-mean):', TAB.round(3).to_string(index=False),
    '', 'Paired hierarchical bootstrap RMSE(V0) - RMSE(model), 100,000 resamples:', TESTS.round(4).to_string(index=False),
    '', 'Secondary, cycle-level uncapped (descriptive):', SEC.round(3).to_string(index=False)])
open(f'{OUT}/e2c_summary.txt', 'w').write(txt)
H = {os.path.basename(p): hashlib.sha256(open(p, 'rb').read()).hexdigest() for p in sorted(glob.glob(f'{OUT}/e2c_*.*'))}
json.dump(H, open(f'{OUT}/sha256_e2c_score.json', 'w'), indent=1); print(txt)